# MMELON Inference

Score molecules from a CSV with a fine-tuned MMELON model from
[MMELON_finetuning.ipynb](MMELON_finetuning.ipynb) (`best_model.ckpt`).
For the k-fold models from [MMELON_finetuning_cv.ipynb](MMELON_finetuning_cv.ipynb), use
[MMELON_inference_cv.ipynb](MMELON_inference_cv.ipynb).

**Environment:** run with the `biomed-multiview` kernel.

**Input:** a CSV or parquet file with a SMILES column. A 0/1 label column is optional; if present, metrics
are reported. Only the SMILES, label and `KEEP_COLS` columns are read.

**Metrics** (only with labels)
- **hit@K** (hit@200, hit@100, hit@50): rank molecules by score; hit@K is how many of the top K have label 1.
- AUROC, PR AUC, and precision, recall, specificity and accuracy at `THRESHOLD`.

**Output**, next to the model in `<model run folder>/predictions/<PREDICTION_NAME>/`:
```
predictions.csv   # every input row (row_index, SMILES, label, KEEP_COLS), plus score, pred and rank
                  # (1 = highest score)
metrics.json      # only if the CSV has labels
config.json       # settings used
```

On CPU, scoring takes roughly 0.25 s per molecule; use a GPU for large files.

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this prediction; outputs go to <model run folder>/predictions/<PREDICTION_NAME>/
PREDICTION_NAME = "test"

# Fine-tuned model: a best_model.ckpt file, or the run folder that contains it
MODEL_PATH = Path("../output") / DATASET / "finetuned" / "mmelon-final_20261020_162634"

# Molecules to score
DATA_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/test.parquet")
SMILES_COL = "SMILES"
LABEL_COL = "LABEL"
KEEP_COLS = []        # other input columns to copy into predictions.csv, e.g. ["ID"]

In [2]:
import os

BATCH_SIZE = 32
NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes: all CPUs, up to 24
THRESHOLD = 0.5           # score above this -> pred = 1
HIT_KS = [200, 100, 50]   # hit@K = number of label-1 molecules among the K highest-scored

In [3]:
import json
import shutil

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())

MODEL_PATH = Path(MODEL_PATH).expanduser().resolve()
if MODEL_PATH.is_dir():
    MODEL_PATH = MODEL_PATH / "best_model.ckpt"
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"

OUT_DIR = MODEL_PATH.parent / "predictions" / PREDICTION_NAME
assert not (OUT_DIR.exists() and any(OUT_DIR.iterdir())), (
    f"{OUT_DIR} already exists. Choose a new PREDICTION_NAME or delete that folder."
)
PIPELINE_DIR = OUT_DIR / "_pipeline"  # temporary copy for the data pipeline, deleted at the end
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

config = dict(model_path=str(MODEL_PATH), data_path=str(DATA_PATH), smiles_col=SMILES_COL,
              label_col=LABEL_COL, threshold=THRESHOLD, hit_ks=HIT_KS)
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Model  : {MODEL_PATH}")
print(f"Data   : {DATA_PATH}")
print(f"Output : {OUT_DIR}")

Model  : ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634/best_model.ckpt
Data   : ~/repos/biomed-multi-view/aircheck/sample_data/test.parquet
Output : ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634/predictions/test


## 2. Prepare data

Every SMILES is checked with RDKit first: one unreadable SMILES would otherwise stop the whole run.
Invalid rows are kept in the output with an empty score. If the CSV has no label column,
predictions are still made and the metrics step is skipped.

In [4]:
import pandas as pd
from rdkit import Chem, RDLogger

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)

RDLogger.DisableLog("rdApp.*")

input_df = read_table(DATA_PATH, [SMILES_COL, *KEEP_COLS], optional=[LABEL_COL] if LABEL_COL else [])
input_df.insert(0, "row_index", range(len(input_df)))  # row in the input file
HAS_LABELS = LABEL_COL is not None and LABEL_COL in input_df.columns and input_df[LABEL_COL].notna().any()

smiles = input_df[SMILES_COL].astype(str).str.strip()
valid = input_df[SMILES_COL].notna() & smiles.map(lambda s: Chem.MolFromSmiles(s) is not None)

# The MMELON pipeline reads data_predict.csv with columns smiles and label (a placeholder if unlabeled)
pipeline_df = pd.DataFrame({
    "smiles": smiles[valid],
    "label": input_df.loc[valid, LABEL_COL].fillna(0).astype(int) if HAS_LABELS else 0,
})
pipeline_df.to_csv(PIPELINE_DIR / "data_predict.csv", index=False)

print(f"Rows           : {len(input_df):,}")
print(f"Valid SMILES   : {valid.sum():,}")
if (~valid).any():
    print(f"Invalid SMILES : {(~valid).sum():,} (kept with an empty score), e.g. {input_df.loc[~valid, SMILES_COL].head(3).tolist()}")
print(f"Labels         : {'yes' if HAS_LABELS else 'no (metrics will be skipped)'}")

Rows           : 1,031
Valid SMILES   : 1,031
Labels         : yes


## 3. Load the data pipeline

Each SMILES is converted into the three MMELON views: text tokens, a 2D image and a molecular graph.

In [5]:
import gc
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from bmfm_sm.core.data_modules.namespace import TaskType
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(PIPELINE_DIR),
    dataset_args={
        "task_type": TaskType.CLASSIFICATION,
        "num_tasks": 1,
        "modalities": ["TEXT_MODEL", "IMAGE_MODEL", "GRAPH_2D_MODEL"],
        "smiles_col": "smiles",
        "label_cols": ["label"],
        "split_col": None,
    },
    stage="predict",
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


def predict(model_path: Path) -> np.ndarray:
    # Score every valid molecule (in data_predict.csv order) with one fine-tuned model
    model = FineTuneLightningModule.load_from_checkpoint(model_path, map_location=DEVICE)
    model.to(DEVICE).eval()
    scores = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=model_path.parent.name):
            batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
            # Encode the batch, then fuse each molecule on its own: the attention aggregator averages its
            # view weights over the batch, so batched fusion would make a score depend on its batch-mates
            separate = model.model.get_separate_outputs(batch)
            per_view = [separate[k] for k in ("Graph2dModel", "ImageModel", "TextModel")]  # aggregator order
            fused = torch.cat([model.model.aggregator([v[i:i + 1] for v in per_view])[0]
                               for i in range(len(per_view[0]))])
            scores.append(torch.sigmoid(model.pred_head(fused)).reshape(-1).cpu())
    del model
    gc.collect()
    scores = torch.cat(scores).numpy()
    assert len(scores) == len(pipeline_df), f"{len(scores)} scores for {len(pipeline_df)} molecules"
    return scores


def compute_metrics(labels, scores) -> dict:
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(labels[top[:k]].sum()) for k in HIT_KS}
    pred = scores > THRESHOLD
    tp, fp = int((pred & (labels == 1)).sum()), int((pred & (labels == 0)).sum())
    tn, fn = int((~pred & (labels == 0)).sum()), int((~pred & (labels == 1)).sum())
    both_classes = len(set(labels)) == 2
    metrics.update(
        rocauc=roc_auc_score(labels, scores) if both_classes else float("nan"),
        pr_auc=average_precision_score(labels, scores) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(labels),
    )
    return metrics


print(f"Device: {DEVICE}; {len(dataset):,} molecules in {len(loader):,} batches")

2026-10-20 16:48:11,545 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 16:48:11,546 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


Device: cuda; 1,031 molecules in 33 batches


## 4. Score the molecules

In [6]:
%%time
scores = predict(MODEL_PATH)

predictions = input_df.copy()
predictions["score"] = np.nan
predictions.loc[valid, "score"] = scores
predictions["pred"] = (predictions["score"] > THRESHOLD).astype("Int64").where(valid)
predictions["rank"] = predictions["score"].rank(ascending=False, method="first").astype("Int64")
predictions.to_csv(OUT_DIR / "predictions.csv", index=False)
shutil.rmtree(PIPELINE_DIR)

print(f"Saved {len(predictions):,} rows to {OUT_DIR / 'predictions.csv'}")
display(predictions.sort_values("rank").head(10))

2026-10-20 16:48:13,436 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - Initializing model with params {'agg_arch': 'coeff_mlp', 'agg_gate_input': 'coeff_mlp', 'agg_weight_freeze': None, 'inference_mode': False}


2026-10-20 16:48:14,697 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - Using coeff_mlp architecture for aggregator


2026-10-20 16:48:14,698 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - dim_list [512, 512, 768] for aggregator


2026-10-20 16:48:14,720 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - Initialized the model with random weights as checkpoint_path was specified as None


2026-10-20 16:48:14,721 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - Task type is classification


...

2026-10-20 16:48:14,979 - root - INFO - instance-shay-ml-01:139714847442752:0:0 - in train False setting deterministic_eval = True


mmelon-final_20261020_162634:   0%|          | 0/33 [00:00<?, ?it/s]

Saved 1,031 rows to ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634/predictions/test/predictions.csv


,row_index,SMILES,LABEL,score,pred,rank
377,377,O=C(Nc1ccccc1)c1cc(-c2cccnc2)nc2ccccc12,1,0.851878,1,1
284,284,CN1C(=O)C(c2c(-c3ccc(Cl)cc3)[nH]c3c2c(=O)[nH]c...,1,0.851531,1,2
591,591,CS(=O)(=O)c1ccc(NC(=O)c2cc(-c3ccncc3)nc3ccccc2...,1,0.850611,1,3
337,337,Cc1cccc(-n2c(SCc3ccn(C)c(=O)c3)nnc2-c2ccc(Cl)c...,1,0.850318,1,4
708,708,Cc1ccc2cc(C(=O)NC(c3ccccc3)c3ccccn3)[nH]c2c1,1,0.850232,1,5
184,184,Cn1ccnc1C(NC(=O)c1cc2cc(F)ccc2s1)c1ccc(Cl)cc1,1,0.849363,1,6
472,472,CC(=O)NCc1ccc(C(=O)NC(c2ccccc2)c2ccc(Cl)cc2)cc1,1,0.849164,1,7
480,480,O=C(Nc1c[nH]nc1-c1ccccc1F)c1ccc2c(c1)CCC2,0,0.847998,1,8
551,551,Cc1ccc(NC(=O)c2n[nH]c(=O)c3ccccc23)c(C)c1,0,0.847469,1,9
581,581,COc1ccc(C(=O)Nc2c(-c3ccccc3)nc3ncccn23)cc1,1,0.847445,1,10


CPU times: user 9.18 s, sys: 2.73 s, total: 11.9 s
Wall time: 1min 34s


## 5. Metrics (if the CSV has labels)

In [7]:
if not HAS_LABELS:
    print("No labels in the CSV; skipping metrics.")
else:
    scored = predictions[valid & predictions[LABEL_COL].notna()]
    metrics = compute_metrics(scored[LABEL_COL], scored["score"])
    metrics = dict(n_molecules=len(scored), n_positives=int(scored[LABEL_COL].sum()), **metrics)
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
    display(pd.Series(metrics, name="value").to_frame().round(4))

,value
n_molecules,1031.0000
n_positives,31.0000
hit@200,23.0000
hit@100,13.0000
hit@50,10.0000
rocauc,0.8197
pr_auc,0.3373
precision,0.1230
recall,0.7419
specificity,0.8360


In [8]:
# Files produced
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUT_DIR)):40s} {path.stat().st_size / 1e6:8.2f} MB")

config.json                                  0.00 MB
metrics.json                                 0.00 MB
predictions.csv                              0.08 MB
